# 04 — Tools

## Objectives

- Give an agent real capabilities with `@agent.tool` and `@agent.tool_plain`
- Let the model's docstring become the tool's description (docstring → schema)
- Say "no" properly with `ModelRetry` from inside a tool
- Prove which tools actually ran by inspecting the messages

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## The idea

A tool is a Python function the model is allowed to *call*. The model never sees your code —
it sees a JSON schema built from your **type hints**, plus a description taken verbatim from
your **docstring**. The model decides *when* to call; your function decides *what happens*.

Two decorators:

- `@agent.tool` — the function takes `RunContext` as its first parameter (that's where
  dependencies live — next notebook). Tools that need no context use…
- `@agent.tool_plain` — plain function, no context.

**Write docstrings like the model will read them** — because it will.

In [ ]:
import random

from pydantic_ai import Agent, RunContext

dice_agent = Agent(
    MODEL,
    instructions="You run a dice game. Use your tools — never invent rolls or names.",
)


@dice_agent.tool_plain
def roll_dice() -> int:
    """Roll a six-sided die and return the result (1-6)."""
    return random.randint(1, 6)


@dice_agent.tool
def get_player_name(ctx: RunContext[None]) -> str:
    """Get the name of the current player."""
    return "Anne"


result = await dice_agent.run("Roll the dice, then tell me whether Anne's guess of 4 beat it.")
print(result.output)

> ✅ **Checkpoint** — the answer references a real roll (your Python `random.randint` ran!).
>
> 🔭 In the Logfire trace you can see the shape of an agentic loop:
> model call → **tool call span** → model call again with the tool's result folded in.

Don't trust it? Verify from the messages themselves — tool executions leave
`ToolReturnPart`s in the history:

In [ ]:
from pydantic_ai.messages import ToolReturnPart

tool_returns = [
    part.tool_name
    for message in result.all_messages()
    for part in message.parts
    if isinstance(part, ToolReturnPart)
]
print("tools that actually ran:", tool_returns)

> ✅ **Checkpoint** — `['roll_dice', 'get_player_name']` (order may vary): proof the model
> called *your functions* and your code actually executed.

## Saying no: `ModelRetry` inside tools

Tools are your boundary of control. If the model calls a tool with bad arguments, don't
crash — raise `ModelRetry` with a helpful message. The model gets the error, corrects
itself, and calls again. Notice this tool is `async def`: tools may be sync or async —
prefer async for anything doing I/O (asyncio rule 3).

In [ ]:
from pydantic_ai import ModelRetry

capitals = {"france": "Paris", "japan": "Tokyo", "kenya": "Nairobi"}

quiz_agent = Agent(
    MODEL,
    instructions="You quiz the user on capitals. Use the lookup tool; never answer from memory.",
)


@quiz_agent.tool
async def lookup_capital(ctx: RunContext[None], country: str) -> str:
    """Look up the capital of a country. The country must exist in our tiny database."""
    key = country.strip().lower()
    if key not in capitals:
        known = ", ".join(sorted(capitals))
        raise ModelRetry(
            f"Unknown country {country!r}. Known countries: {known}. Try one of those."
        )
    return capitals[key]


r = await quiz_agent.run("What's the capital of the country famous for the Eiffel Tower?")
print(r.output)

> ✅ **Checkpoint** — the model had to reason "Eiffel Tower → France", call the tool, and get
> Paris back. If you ask it something like "capital of Ruritania?", you'll see the `ModelRetry`
> rejection loop live — and find it in the Logfire trace.

## 🏋️ Exercise — the temperature agent

Build `temp_agent` with two `@agent.tool_plain` tools: `c_to_f(celsius: float) -> float` and
`f_to_c(fahrenheit: float) -> float` (exact formulas — no model arithmetic!). Then ask:

> "Is 25°C warmer than 80°F?"

The agent *must* convert both to one scale using your tools. Prove it used them by scanning
`result.all_messages()` for `ToolReturnPart`s (the pattern is two cells above). If your agent
tried to do the math itself, harden the `instructions` until it behaves.

Your take: [`../solutions/04_temperature_tools.ipynb`](../solutions/04_temperature_tools.ipynb).

## Next

→ **[05_dependencies.ipynb](05_dependencies.ipynb)** — what `RunContext` is actually carrying.